# 04 — Inference Optimization: Cache, Batch & Streaming

**Vai trò:** Model Engineer · **Task:** S4-ME-03 (Yêu cầu 9.4)

OLLAMA chạy **cục bộ** (constraint cốt lõi của dự án) nên tốc độ suy luận phụ thuộc trực tiếp vào phần cứng máy người dùng. Ở tầng client (`OllamaEmbeddingModel`/`OllamaClient`, S2-ME-*, S3-ME-*), có ba kỹ thuật tối ưu suy luận thực tế mà ta có thể đo lường và áp dụng ngay: **cache** (tránh gọi lại OLLAMA cho input đã thấy), **batch** (đo thông lượng khi xử lý nhiều đoạn cùng lúc), và **streaming** (giảm độ trễ cảm nhận bằng cách hiển thị token ngay khi có, qua `generate_stream()` — S3-ME-02).

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.generation.llm_client import OllamaClient
from src.pipeline.experiment_tracker import ExperimentTracker

client = OllamaClient(model_name="llama3", max_tokens=120)
embedder = OllamaEmbeddingModel(model_name="nomic-embed-text")
available = client.is_available()
print(f"Project root: {PROJECT_ROOT}")
print(f"OLLAMA kha dung tai {client.base_url}: {available}")
if not available:
    print(
        "\n⚠️  OLLAMA chua san sang. Hay chay 'ollama serve' va pull cac model "
        "'llama3'/'nomic-embed-text' roi chay lai notebook de do thuc."
    )

## 1. Cache — tránh gọi lại OLLAMA cho cùng một đầu vào

`embed_text()` là **tất định** (Property 3/4) — cùng một văn bản luôn cho cùng vector, nên kết quả có thể **cache an toàn**. Một cache từ điển đơn giản giúp các lượt gọi lặp lại (vd. cùng câu hỏi được hỏi nhiều lần) trả về ngay lập tức thay vì gửi request HTTP mới tới OLLAMA.

In [ ]:
class CachedEmbedder:
    # Boc OllamaEmbeddingModel voi mot cache tu dien don gian theo van ban dau vao.

    def __init__(self, model):
        self._model = model
        self._cache = {}
        self.hits = 0
        self.misses = 0

    def embed_text(self, text):
        if text in self._cache:
            self.hits += 1
            return self._cache[text]
        self.misses += 1
        vector = self._model.embed_text(text)
        self._cache[text] = vector
        return vector


cached_embedder = CachedEmbedder(embedder)
probe_text = "RAG ket hop retrieval va generation de tra loi cau hoi dua tren tai lieu rieng."

cache_timings = []
for label in ["lan 1 (cache MISS)", "lan 2 (cache HIT)", "lan 3 (cache HIT)"]:
    start = time.perf_counter()
    vector = cached_embedder.embed_text(probe_text)
    elapsed_ms = (time.perf_counter() - start) * 1000
    cache_timings.append((label, elapsed_ms))
    print(f"{label}: {elapsed_ms:.2f} ms  (chieu vector={len(vector)})")

print(f"\nCache stats: {cached_embedder.hits} hit / {cached_embedder.misses} miss")
if cache_timings[0][1] > 0:
    speedup = cache_timings[0][1] / max(cache_timings[1][1], 1e-6)
    print(f"Toc do tang toc tu cache (lan 1 / lan 2): ~{speedup:.1f}x")

## 2. Batch — thông lượng khi nhúng nhiều đoạn cùng lúc

`embed_batch()` (Property 5, S2-ME-02) xử lý cả lô văn bản trong một lệnh gọi — notebook đo **thông lượng** (số đoạn/giây) để người học hình dung chi phí nhúng một tài liệu lớn trước khi index (`RAGPipeline.index_document()`, pseudocode §2.7).

In [ ]:
batch_texts = [f"Day la doan van ban thu nghiem so {i} de do thong luong embedding." for i in range(8)]

start = time.perf_counter()
batch_vectors = embedder.embed_batch(batch_texts)
batch_latency_ms = (time.perf_counter() - start) * 1000
throughput = len(batch_texts) / (batch_latency_ms / 1000)

assert len(batch_vectors) == len(batch_texts)
print(f"Da nhung {len(batch_texts)} doan trong {batch_latency_ms:.1f} ms")
print(f"Thong luong: ~{throughput:.2f} doan/giay")
print(f"Latency trung binh moi doan: {batch_latency_ms / len(batch_texts):.1f} ms")

## 3. Streaming — giảm độ trễ cảm nhận với `generate_stream()`

`generate()` chờ toàn bộ câu trả lời rồi mới trả về một lần; `generate_stream()` (S3-ME-02, Yêu cầu 5.3) **yield từng token ngay khi sinh ra**. Với người dùng cuối, điều quan trọng không chỉ là tổng thời gian mà còn là **thời gian tới token đầu tiên** (time-to-first-token) — đây là phép đo cho thấy vì sao streaming cải thiện trải nghiệm cảm nhận dù tổng thời gian xử lý tương đương.

In [ ]:
PROMPT = "Liet ke ngan gon 3 loi ich cua kien truc RAG so voi chi dung LLM thuan tuy."

if available:
    # Bloc: doi toan bo cau tra loi
    start = time.perf_counter()
    full_answer = client.generate(PROMPT)
    blocking_total_ms = (time.perf_counter() - start) * 1000

    # Streaming: do thoi gian den token dau tien va tong thoi gian
    start = time.perf_counter()
    first_token_ms = None
    streamed_chunks = []
    for token in client.generate_stream(PROMPT):
        if first_token_ms is None:
            first_token_ms = (time.perf_counter() - start) * 1000
        streamed_chunks.append(token)
    streaming_total_ms = (time.perf_counter() - start) * 1000
    streamed_answer = "".join(streamed_chunks)

    print(f"[blocking ] tong thoi gian = {blocking_total_ms:.0f} ms  ({len(full_answer)} ky tu)")
    print(f"[streaming] token dau tien sau = {first_token_ms:.0f} ms")
    print(f"[streaming] tong thoi gian = {streaming_total_ms:.0f} ms  ({len(streamed_answer)} ky tu)")
    print(
        f"\n=> Streaming cho phep hien thi noi dung som hon "
        f"~{max(blocking_total_ms - first_token_ms, 0):.0f} ms so voi cho ca cau tra loi — "
        f"day la ly do app/components/chat_widget.py uu tien hien thi tung token."
    )
    assert streamed_answer, "generate_stream phai sinh duoc cau tra loi khong rong"
else:
    blocking_total_ms = first_token_ms = streaming_total_ms = 0.0
    print("Bo qua do luong streaming — OLLAMA khong kha dung (xem canh bao o muc dau).")

## 4. Ghi lại thực nghiệm & tổng kết

Mỗi kỹ thuật tối ưu được ghi lại như một sự kiện "query" qua `ExperimentTracker.log_query()` (Yêu cầu 9.8) — `params["question"]` đánh dấu rõ kỹ thuật để dễ lọc lại khi xem ở trang Experiment Log.

In [ ]:
tracker = ExperimentTracker()
tracker.log_query(
    question="[inference_optimization:cache] embed_text lap lai",
    top_k=0, contexts=[], answer=f"{cached_embedder.hits} hit / {cached_embedder.misses} miss",
    latency_ms=cache_timings[0][1],
)
tracker.log_query(
    question="[inference_optimization:batch] embed_batch throughput",
    top_k=0, contexts=[], answer=f"~{throughput:.2f} doan/giay",
    latency_ms=batch_latency_ms,
)
if available:
    tracker.log_query(
        question="[inference_optimization:streaming] time-to-first-token",
        top_k=0, contexts=[], answer=f"first_token={first_token_ms:.0f}ms",
        latency_ms=streaming_total_ms,
    )

print("Tom tat phien thuc nghiem:")
print(tracker.get_summary())

## 5. Tổng kết

- **Cache**: vì embedding là **tất định** (Property 3/4), việc cache theo văn bản đầu vào là an toàn và loại bỏ hoàn toàn round-trip HTTP cho các lượt lặp lại — hữu ích khi cùng một câu hỏi/chunk được xử lý nhiều lần.
- **Batch**: `embed_batch()` cho thấy thông lượng (đoạn/giây) khi xử lý nhiều đoạn — số liệu này giúp ước lượng thời gian `index_document()` cho một tài liệu lớn trước khi chạy thật.
- **Streaming**: `generate_stream()` không làm giảm tổng thời gian xử lý, nhưng giảm đáng kể **thời gian tới token đầu tiên** — cải thiện trải nghiệm cảm nhận, đúng lý do `RAGPipeline.query_stream()` và `chat_widget.py` ưu tiên hiển thị theo dòng token.
- Cả ba phép đo được ghi lại qua `ExperimentTracker` mà không làm gián đoạn notebook (Yêu cầu 9.8).